# B2.5 · Model tiering and routing inside the loop

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

---

**Risk.** Paying frontier prices for executor-grade steps.

**Control.** Cheap executor, escalated reasoner, advisor at decision points.

**This lab.** Route cheap executor to escalated reasoner and attribute spend.

| | |
|---|---|
| Open-source tooling | LiteLLM, vLLM |
| Open-weight models | Llama 3.3, GLM-4.6, Kimi K2 |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B2.5"))

Model tiering inside the loop is where cost optimisation quietly becomes a security decision.

In [ ]:
from cybercommons import loop, planes

# route by task, but attach tools by blast radius
ROUTES = {
    "plan":   ("Kimi K2 (large)",   planes.Manifest("plan",   [planes.Tool("read_file")], rung="L1")),
    "act":    ("GLM-4.6 (mid)",     planes.Manifest("act",
                  [planes.Tool("read_file"),
                   planes.Tool("write_file", writes=True, scope="project")],
                  approval_required={"write_file"}, rung="L2")),
    "verify": ("Llama 3.3 (small)", planes.Manifest("verify", [planes.Tool("read_file")], rung="L1")),
}
for stage, (model, m) in ROUTES.items():
    print(f"{stage:7s} {model:20s} blast={m.blast_radius()['total']:3d} "
          f"issues={m.rung_check() or 'none'}")

Now the anti-pattern, priced honestly: put the tools on the cheap fast model so the loop feels responsive.

In [ ]:
bad = planes.Manifest("cheap-actor", [
    planes.Tool("read_file"),
    planes.Tool("write_file",  writes=True, scope="project"),
    planes.Tool("deploy_prod", writes=True, scope="org", reversible=False),
], rung="L2.5")
print("cheap model holding the tools → blast", bad.blast_radius()["total"])
for p in bad.rung_check():
    print("  ⚠", p)
print("\nThe saving is real. So is putting the weakest reasoning next to the")
print("highest authority. Tiering is a routing decision AND an authority decision.")

### Expect

All three staged routes report a blast radius of 0 with no rung problems. The anti-pattern scores 43 and flags an irreversible ungated org-wide tool.

### Your turn

Verification is the stage most often given to the cheapest model. Given B2.2, argue whether that is defensible — and what property the verifier model needs that the planner does not.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B2.5.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*